# Transcription in Google Colab
This notebook shows how to load the GEMMA-3n model with a LoRA adapter, upload an audio file, and get a transcription.

In [ ]:
!pip install sounddevice numpy torch transformers peft bitsandbytes unsloth soundfile librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.3/299.3 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.5/166.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 127.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23

In [ ]:
import queue
import numpy as np
import torch
from peft import PeftModel
from transformers import TextStreamer
from unsloth import FastModel
import soundfile as sf
import librosa

/tmp/ipython-input-2-1070086345.py:6: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from huggingface_hub import notebook_login; notebook_login()

In [ ]:
# Load model and processor via FastModel
base_model = "unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit"
lora_adapter = "KronosDP/gemma-3n-id-4bit-300v"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading model and processor…")
model, processor = FastModel.from_pretrained(
    model_name=base_model,
    dtype=None,
    max_seq_length=1024,
    load_in_4bit=True,
    full_finetuning=False
)
model = PeftModel.from_pretrained(model, lora_adapter, is_trainable=False)
model.to(device)
model.eval()

Loading model and processor…
==((====))==  Unsloth 2025.7.8: Fast Gemma3N patching. Transformers: 4.53.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3N does not support SDPA - switching to eager!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/469M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/50.3M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3nForConditionalGeneration(
      (model): Gemma3nModel(
        (vision_tower): TimmWrapperModel(
          (timm_model): MobileNetV5Encoder(
            (conv_stem): ConvNormAct(
              (conv): Conv2dSame(3, 64, kernel_size=(3, 3), stride=(2, 2))
              (bn): RmsNormAct2d(
                (drop): Identity()
                (act): GELU(approximate='tanh')
              )
            )
            (blocks): Sequential(
              (0): Sequential(
                (0): EdgeResidual(
                  (conv_exp): Conv2dSame(64, 256, kernel_size=(3, 3), stride=(2, 2), bias=False)
                  (bn1): RmsNormAct2d(
                    (drop): Identity()
                    (act): GELU(approximate='tanh')
                  )
                  (aa): Identity()
                  (se): Identity()
                  (conv_pwl): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
                  (

## Upload an Audio File
Use the file-upload widget below to select a WAV/FLAC/MP3 file from your local machine.

In [ ]:
from google.colab import files
import soundfile as sf
import librosa

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {audio_path}")

# Save the uploaded file to disk
with open(audio_path, "wb") as f:
    f.write(uploaded[audio_path])

# Read and prepare audio
print(audio_path)
audio_array, sr = librosa.load(audio_path, sr=16000)
audio_list = audio_array.flatten().tolist()

Saving inference_audio.mp3 to inference_audio (1).mp3
Uploaded: inference_audio (1).mp3
inference_audio (1).mp3


/tmp/ipython-input-29-2527046152.py:15: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, sr = librosa.load(audio_path, sr=16000)
/usr/local/lib/python3.11/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [ ]:
from IPython.display import Audio
Audio(audio_list, rate=sr)

Real text: Halo semuanya, ini adalah testing untuk inference script dari Gemma.

In [ ]:
# Inference helper
def do_gemma_3n_inference(audio_list, max_new_tokens=128):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are an assistant that transcribes speech accurately."}]},
        {"role": "user", "content": [{"type": "audio", "audio": audio_list}, {"type": "text", "text": "Please transcribe this audio."}]}
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)
    streamer = TextStreamer(processor, skip_prompt=True)
    _ = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        streamer=streamer
    )

print("Transcription:")
do_gemma_3n_inference(audio_array, max_new_tokens=512)

Transcription:
Halo semuanya, ini adalah testing untuk inference script dari Gema.<end_of_turn>
